In [0]:
%sql
CREATE TABLE IF NOT EXISTS pondolens.dev_silver.regional_appropriations (
  gaa_id INT NOT NULL,
  year INT,
  category STRING,
  particular STRING,
  region STRING,
  appropriation_amount DECIMAL(20,2),
  processed_at TIMESTAMP,
  CONSTRAINT regional_appropriations_pk PRIMARY KEY (gaa_id)
)
USING DELTA;

-- Trim text, cast year and remove identical cleaned records.
CREATE OR REPLACE TEMP VIEW regional_gaa_stage AS
SELECT DISTINCT
  CAST(TRIM(CAST(year AS STRING)) AS INT) AS year,
  NULLIF(TRIM(category), '') AS category,
  NULLIF(TRIM(particulars), '') AS particular,

  -- Standardize Region-IVA and Region IV-A to Region IVA.
  CASE
    WHEN TRIM(regions) RLIKE r'(?i)^region[\s-]*([ivx]+)[\s-]*([ab]?)$'
    THEN CONCAT(
      'Region ',
      UPPER(REGEXP_EXTRACT(
        TRIM(regions),
        r'(?i)^region[\s-]*([ivx]+)[\s-]*([ab]?)$',
        1
      )),
      UPPER(REGEXP_EXTRACT(
        TRIM(regions),
        r'(?i)^region[\s-]*([ivx]+)[\s-]*([ab]?)$',
        2
      ))
    )
    WHEN LOWER(TRIM(regions)) = 'nationawide' THEN 'Nationwide'
    ELSE NULLIF(TRIM(regions), '')
  END AS region,

  -- Dashes become zero; blanks stay null. Convert thousand pesos to pesos.
  CASE
    WHEN TRIM(CAST(expenditure_amount AS STRING)) = '-'
      THEN CAST(0 AS DECIMAL(20,2))
    WHEN expenditure_amount IS NULL
      OR TRIM(CAST(expenditure_amount AS STRING)) = ''
      THEN CAST(NULL AS DECIMAL(20,2))
    ELSE CAST(
      CAST(
        REPLACE(TRIM(CAST(expenditure_amount AS STRING)), ',', '')
        AS DECIMAL(20,2)
      ) * 1000
      AS DECIMAL(20,2)
    )
  END AS appropriation_amount
FROM pondolens.dev_bronze.regional_expenditure_gaa;

-- Stop on missing keys or conflicting source records.
SELECT ASSERT_TRUE(
  COUNT(*) = COUNT(year)
  AND COUNT(*) = COUNT(category)
  AND COUNT(*) = COUNT(region)
  AND COUNT(*) = COUNT(
    DISTINCT STRUCT(year, category, particular, region)
  ),
  'Missing key fields or conflicting source records.'
)
FROM regional_gaa_stage;

SELECT ASSERT_TRUE(
  COUNT(*) = COUNT(DISTINCT gaa_id)
  AND COUNT(*) = COUNT(year)
  AND COUNT(*) = COUNT(category)
  AND COUNT(*) = COUNT(region)
  AND COUNT(*) = COUNT(
    DISTINCT STRUCT(year, category, particular, region)
  ),
  'Missing keys or duplicate records in the Silver table.'
)
FROM pondolens.dev_silver.regional_appropriations;

CREATE OR REPLACE TEMP VIEW regional_gaa_numbered AS
WITH new_records AS (
  -- Only records not already in Silver need a new ID.
  SELECT s.*
  FROM regional_gaa_stage AS s
  LEFT ANTI JOIN pondolens.dev_silver.regional_appropriations AS t
    ON s.year = t.year
    AND s.category = t.category
    AND s.particular <=> t.particular
    AND s.region = t.region
),
last_sequence AS (
  SELECT
    year,
    MAX(gaa_id - CAST(year AS BIGINT) * 100000) AS last_number
  FROM pondolens.dev_silver.regional_appropriations
  GROUP BY year
)
SELECT
  s.*,
  COALESCE(l.last_number, 0)
    + ROW_NUMBER() OVER (
        PARTITION BY s.year
        ORDER BY s.category, s.particular, s.region
      ) AS sequence_number
FROM new_records AS s
LEFT JOIN last_sequence AS l
  ON s.year = l.year;

-- Check the sequence before padding it to five digits.
SELECT ASSERT_TRUE(
  COUNT(*) = 0,
  'Sequence must be between 1 and 99999; year must have four digits.'
)
FROM regional_gaa_numbered
WHERE sequence_number NOT BETWEEN 1 AND 99999
  OR year NOT BETWEEN 1000 AND 9999;

CREATE OR REPLACE TEMP VIEW regional_gaa_merge_source AS
-- Keep the IDs of existing records.
SELECT
  CAST(t.gaa_id AS INT) AS gaa_id,
  s.year,
  s.category,
  s.particular,
  s.region,
  s.appropriation_amount
FROM regional_gaa_stage AS s
INNER JOIN pondolens.dev_silver.regional_appropriations AS t
  ON s.year = t.year
  AND s.category = t.category
  AND s.particular <=> t.particular
  AND s.region = t.region

UNION ALL

-- Build year + five-digit sequence, then store it as INT.
SELECT
  CAST(
    CONCAT(
      CAST(year AS STRING),
      LPAD(CAST(sequence_number AS STRING), 5, '0')
    )
    AS INT
  ) AS gaa_id,
  year,
  category,
  particular,
  region,
  appropriation_amount
FROM regional_gaa_numbered;

MERGE INTO pondolens.dev_silver.regional_appropriations AS t
USING regional_gaa_merge_source AS s
ON t.year = s.year
  AND t.category = s.category
  AND t.particular <=> s.particular
  AND t.region = s.region

-- Leave unchanged amounts and timestamps untouched.
WHEN MATCHED
  AND NOT (t.appropriation_amount <=> s.appropriation_amount)
THEN UPDATE SET
  t.appropriation_amount = s.appropriation_amount,
  t.processed_at = CURRENT_TIMESTAMP()

WHEN NOT MATCHED THEN INSERT (
  gaa_id,
  year,
  category,
  particular,
  region,
  appropriation_amount,
  processed_at
)
VALUES (
  s.gaa_id,
  s.year,
  s.category,
  s.particular,
  s.region,
  s.appropriation_amount,
  CURRENT_TIMESTAMP()
);